# Agente

### El ciclo del agente
> **Percibir** (lee la situación) → **Razonar** (decide qué hacer) → **Actuar** (usa una herramienta) → **Observar** (mira el resultado) → y vuelta a empezar hasta tener la respuesta.



## Configuración

Lo de siempre: instalar, clave y cliente. (Si vienes del Notebook 2, es idéntico.)

In [10]:
from openai import OpenAI
from getpass import getpass
import json, datetime

API_KEY = getpass("Pega aquí tu clave API: ")

BASE_URL = "https://api.groq.com/openai/v1"
MODELO   = "llama-3.3-70b-versatile"

cliente = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print("Cliente listo")

Cliente listo


## El problema: un LLM solo no llega a todo

Vamos a comprobar las limitaciones con dos ejemplos. Ejecuta y observa.

In [11]:
def preguntar_simple(texto):
    r = cliente.chat.completions.create(model=MODELO, messages=[{"role": "user", "content": texto}])
    return r.choices[0].message.content

print("Pregunta 1:", preguntar_simple("¿Qué hora es exactamente ahora mismo?"))
print()
print("Pregunta 2:", preguntar_simple("¿Cuánto es 48273 * 9912? Dame solo el número."))

Pregunta 1: No puedo proporcionar la hora exacta en este momento. Sin embargo, puedo sugerirte algunas formas de averiguar la hora actual. Puedes consultar un reloj físico o digital cerca de ti, o buscar en línea "hora actual" junto con tu ubicación para obtener la información más precisa.

Pregunta 2: 479305956


Seguramente la **hora** se la ha inventado (no tiene reloj) y el **cálculo** puede estar mal (los LLM no son calculadoras). 

La solución no es pedirle que se esfuerce más: es **darle herramientas**.

## Creamos las herramientas (funciones de Python)

Una **herramienta** es simplemente una función normal de Python que hace algo bien: calcular, consultar la hora, buscar en una base de datos... 

Vamos a crear tres herramientas sencillas:
1. `calculadora` — hace cuentas exactas.
2. `hora_actual` — devuelve la fecha y hora reales.
3. `buscar_producto` — consulta un pequeño catálogo (simulado) de una tienda.



In [ ]:
import ast, operator

# calculadora segura
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg,
}
def _eval(nodo):
    if isinstance(nodo, ast.Constant) and isinstance(nodo.value, (int, float)):
        return nodo.value
    if isinstance(nodo, ast.BinOp):
        return _OPS[type(nodo.op)](_eval(nodo.left), _eval(nodo.right))
    if isinstance(nodo, ast.UnaryOp):
        return _OPS[type(nodo.op)](_eval(nodo.operand))
    raise ValueError("Operación no permitida")

def calculadora(operacion):
    """Calcula una operación matemática. Ej: '48273 * 9912'."""
    try:
        return str(_eval(ast.parse(operacion, mode="eval").body))
    except Exception:
        return "Error: operación no válida"

#  hora actual 
def hora_actual(_=None):
    
    return datetime.datetime.now().strftime("%A %d/%m/%Y, %H:%M:%S")

#  catálogo simulado de una tienda
CATALOGO = {
    "portátil": "Portátil AeroBook 14\" — 799 €, en stock (12 unidades)",
    "ratón":    "Ratón inalámbrico SilentClick — 19 €, en stock (40 unidades)",
    "monitor":  "Monitor 27\" 144Hz — 229 €, agotado",
}
def buscar_producto(nombre):
    """Busca un producto en el catálogo de la tienda."""
    nombre = (nombre or "").lower()
    for clave, info in CATALOGO.items():
        if clave in nombre:
            return info
    return "No se encontró ese producto en el catálogo."


print(calculadora("48273 * 9912"))
print(hora_actual())
print(buscar_producto("quiero un portátil"))

478481976
Thursday 25/06/2026, 23:43:33
Portátil AeroBook 14" — 799 €, en stock (12 unidades)


De momento **somos nosotros** quienes las llamamos. El objetivo es que **el modelo decida** cuándo usar cada una. Para eso necesitamos un "registro" de herramientas y enseñarle a pedirlas.

In [14]:
HERRAMIENTAS = {
    "calculadora":     calculadora,
    "hora_actual":     hora_actual,
    "buscar_producto": buscar_producto,
}
print("Herramientas:", list(HERRAMIENTAS.keys()))

Herramientas: ['calculadora', 'hora_actual', 'buscar_producto']


## Enseñamos al modelo a pedir herramientas

Esta es la idea central. En el mensaje `system` le explicamos al modelo:
- **Qué herramientas tiene** y para qué sirve cada una.
- **Cómo pedirlas:** respondiendo con un **JSON** muy concreto.

Le pedimos que responda siempre con uno de estos dos formatos:

- Para **usar una herramienta**:
  `{"accion": "usar_herramienta", "herramienta": "nombre", "argumento": "..."}`
- Para **dar la respuesta final** al usuario:
  `{"accion": "responder", "respuesta": "..."}`

Así, leyendo su JSON, sabremos si quiere actuar o si ya ha terminado.

In [15]:
INSTRUCCIONES_AGENTE = """Eres un asistente que puede usar herramientas para responder mejor.

Tienes estas herramientas:
- "calculadora": hace cálculos matemáticos exactos. El argumento es la operación, ej: "23*5+1".
- "hora_actual": devuelve la fecha y hora actuales. No necesita argumento (usa "").
- "buscar_producto": consulta el catálogo de la tienda. El argumento es el nombre del producto.

REGLAS:
- Responde SIEMPRE con un único JSON válido, sin texto adicional ni ```.
- Si necesitas una herramienta, usa:
  {"accion": "usar_herramienta", "herramienta": "<nombre>", "argumento": "<texto>"}
- Cuando ya tengas la información para responder al usuario, usa:
  {"accion": "responder", "respuesta": "<tu respuesta final en lenguaje natural>"}
- Usa herramientas para horas, cálculos o datos de la tienda. No te inventes esos datos.
"""

In [16]:
def extraer_json(texto):
    """Saca un objeto JSON de un texto aunque venga con ruido alrededor."""
    if not texto:
        return None
    inicio, fin = texto.find("{"), texto.rfind("}")
    if inicio == -1 or fin == -1:
        return None
    try:
        return json.loads(texto[inicio:fin + 1])
    except json.JSONDecodeError:
        return None

## El bucle del agente

Aquí juntamos todo. El agente:
1. **Percibe**: lee la conversación.
2. **Razona**: el modelo decide (devuelve un JSON).
3. Si pide una herramienta → la **ejecutamos** (Actuar) y le devolvemos el resultado (Observar). Volvemos al paso 2.
4. Si da la respuesta final → terminamos.

Ponemos un **límite de pasos** (`max_pasos`) para que nunca se quede en un bucle infinito. La opción `verbose=True` nos deja **ver cómo piensa** por dentro.

In [17]:
def agente(pregunta_usuario, max_pasos=5, verbose=True):
    mensajes = [
        {"role": "system", "content": INSTRUCCIONES_AGENTE},
        {"role": "user",   "content": pregunta_usuario},
    ]

    for paso in range(1, max_pasos + 1):
        r = cliente.chat.completions.create(model=MODELO, messages=mensajes, temperature=0)
        salida = r.choices[0].message.content
        decision = extraer_json(salida)

        if decision is None:
            return salida

        mensajes.append({"role": "assistant", "content": salida})

        accion = decision.get("accion")

        if accion == "responder":
            if verbose:
                print(f"[Paso {paso}] El agente responde.")
            return decision.get("respuesta", "(sin respuesta)")

        elif accion == "usar_herramienta":
            nombre = decision.get("herramienta")
            argumento = decision.get("argumento", "")
            funcion = HERRAMIENTAS.get(nombre)

            if funcion is None:
                resultado = f"Error: la herramienta '{nombre}' no existe."
            else:
                resultado = funcion(argumento)

            if verbose:
                print(f"[Paso {paso}] Usa '{nombre}'({argumento!r}) → {resultado}")

            # le devolvemos el resultado para que siga razonando 
            mensajes.append({
                "role": "user",
                "content": f'Resultado de la herramienta {nombre}: {resultado}'
            })
        else:
            return f"No entendí la acción del modelo: {decision}"

    return "He alcanzado el límite de pasos sin terminar."

print("Agente definido")

Agente definido


## ¡Probamos el agente!

Vamos a hacerle las mismas preguntas del principio. Ahora debería **usar las herramientas** en lugar de inventar. 

In [18]:
print("PREGUNTA: ¿Qué hora es?")
print("RESPUESTA:", agente("¿Qué hora es exactamente?"))

PREGUNTA: ¿Qué hora es?
[Paso 1] Usa 'hora_actual'('') → Thursday 25/06/2026, 23:48:25
[Paso 2] El agente responde.
RESPUESTA: Son las 23:48:25 del jueves 25 de junio de 2026.


In [19]:
print("PREGUNTA: ¿Cuánto es 48273 * 9912?")
print("RESPUESTA:", agente("¿Cuánto es 48273 * 9912?"))

PREGUNTA: ¿Cuánto es 48273 * 9912?
[Paso 1] Usa 'calculadora'('48273*9912') → 478481976
[Paso 2] El agente responde.
RESPUESTA: El resultado de la multiplicación es 478481976


In [20]:
print("PREGUNTA: ¿Tenéis monitores? ¿A qué precio?")
print("RESPUESTA:", agente("¿Tenéis monitores en la tienda y a qué precio?"))

PREGUNTA: ¿Tenéis monitores? ¿A qué precio?
[Paso 1] Usa 'buscar_producto'('monitores') → Monitor 27" 144Hz — 229 €, agotado
[Paso 2] El agente responde.
RESPUESTA: Tenemos un monitor de 27 pulgadas con una frecuencia de refresco de 144Hz, pero lamentablemente está agotado. Su precio es de 229 euros.


### Lo interesante: combinar herramientas

Un buen agente puede **encadenar varias herramientas** para una sola pregunta. Probemos algo que necesita la tienda **y** la calculadora a la vez.

In [23]:
print("PREGUNTA: ¿Cuánto costarían 3 monitores?")
print("RESPUESTA:", agente("¿Cuánto me costarían 3 monitores de la tienda en total?"))

PREGUNTA: ¿Cuánto costarían 3 monitores?
[Paso 1] Usa 'buscar_producto'('monitor') → Monitor 27" 144Hz — 229 €, agotado
[Paso 2] Usa 'calculadora'('3*229') → 687
[Paso 3] El agente responde.
RESPUESTA: Te costarían 687 euros en total


Si todo va bien, el agente habrá: (1) buscado el ratón en el catálogo, (2) visto que cuesta 19 €, (3) usado la calculadora para `19 * 3`, y (4) respondido 57 €. **¡Eso es razonar paso a paso!**

(Los modelos gratuitos a veces fallan o se saltan un paso. Si pasa, vuelve a ejecutar: forma parte de aprender a trabajar con ellos.)

## Nota: el "function calling" nativo

Lo que has construido es un agente **a mano**, y es la mejor forma de **entender** cómo funciona por dentro.

En la práctica, muchos modelos (Groq y varios de OpenRouter) ofrecen una función llamada **"function calling" / "tool calling"** nativa: en lugar de pedir el JSON por el prompt, le pasas las herramientas en un parámetro `tools=[...]` y el modelo te devuelve directamente qué función llamar.

Es más cómodo, pero el **concepto es exactamente el mismo** que acabas de aprender: el modelo decide, tú ejecutas, le devuelves el resultado. Lo veremos en los próximos días. Saber hacerlo "a mano" te servirá con **cualquier** modelo, incluso los que no soportan la versión nativa.